# TFM Tenerife — Pipeline completo de TripAdvisor (silver.tripadvisor_ubicaciones / silver.tripadvisor_resenas)

Notebook único y definitivo: recorre los 31 municipios de Tenerife, busca hoteles y restaurantes en la API Terra de TripAdvisor, valida geográficamente cada resultado contra la geometría real de la isla, y sube el conjunto limpio a Bronze (Blob Storage) y Silver (Postgres/PostGIS).

Reúne en un único sitio todo lo aprendido durante el desarrollo (ver notas al final), en vez de dejarlo repartido en varios notebooks sucesivos.

**Resultado esperado al ejecutarlo:** 750 ubicaciones en 30 de los 31 municipios. El Rosario se queda fuera — no tiene página de destino propia en TripAdvisor (confirmado a mano), así que cualquier búsqueda de "El Rosario" encuentra homónimos de fuera de Tenerife que la validación geográfica descarta correctamente.

## Paso 1 — Instalar dependencias

In [ ]:
!pip install -q requests geopandas sqlalchemy psycopg2-binary geoalchemy2 shapely azure-storage-blob python-dotenv pandas

## Paso 2 — Conectar (Postgres/PostGIS + Blob Storage)

In [ ]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from azure.storage.blob import BlobServiceClient

load_dotenv()

engine = create_engine(os.environ['AZURE_DB_URL'], pool_pre_ping=True, pool_recycle=280)
blob_service = BlobServiceClient.from_connection_string(os.environ['AZURE_STORAGE_CONNECTION_STRING'])

CONTENEDOR_BRONZE = 'bronce-raw'
container_client = blob_service.get_container_client(CONTENEDOR_BRONZE)

BASE_URL = 'https://terra.tripadvisor.com/api'
HEADERS = {'X-API-Key': os.environ['TRIPADVISOR_KEY']}

with engine.connect() as conn:
    version = conn.execute(text('SELECT postgis_version();')).scalar()
print('Conectado. PostGIS:', version)
print('Contenedor', CONTENEDOR_BRONZE, 'existe:', container_client.exists())

## Paso 3 — Cargar la geometría real de Tenerife

Es la única defensa fiable contra resultados de fuera de la isla que la búsqueda de texto puede colar (confirmado: sin esto, aparecían hoteles de Italia, Guatemala, Argentina e incluso de Gran Canaria).

In [ ]:
import geopandas as gpd

tenerife_union = gpd.read_postgis(
    'SELECT ST_Union(geometry) AS geometry FROM silver.limites_municipales', engine, geom_col='geometry'
).to_crs(epsg=4326).geometry.iloc[0]

print('Geometria de Tenerife cargada correctamente.')

## Paso 4 — Funciones de búsqueda

**Decisiones clave, ya incorporadas:**
- `geo_name` combinado con `category=HOTEL` da un error 503 reproducible en el servidor de Terra (confirmado en múltiples municipios y con distintos parámetros, incluido `address`). Para HOTEL se usa directamente el método de texto, sin perder tiempo intentando `geo_name`.
- `RESTAURANT` sí usa `geo_name` con normalidad — ahí funciona bien.
- `country_code='ES'` reduce ruido obvio de fuera de España en la búsqueda de texto (no es la protección real, esa la da el filtro geográfico del Paso 6).
- Varios municipios no se encuentran bien con su nombre oficial completo porque TripAdvisor los guarda con un nombre distinto (sin tildes/diéresis, o acortado) — ver `NOMBRES_CORTOS`.

In [ ]:
import time
import requests
from shapely.geometry import Point

peticiones_realizadas = 0


def peticion_con_reintentos(url, params, intentos_max=5):
    global peticiones_realizadas
    espera = 2
    codigos_reintentables = (429, 500, 502, 503, 504)
    for intento in range(intentos_max):
        peticiones_realizadas += 1
        resp = requests.get(url, headers=HEADERS, params=params, timeout=30)
        if resp.status_code not in codigos_reintentables:
            resp.raise_for_status()
            return resp
        retry_after = resp.headers.get('Retry-After')
        espera_real = int(retry_after) if retry_after else espera
        print('    error', resp.status_code, '- esperando', espera_real, 'segundos (intento', intento + 1, 'de', intentos_max, ')...')
        time.sleep(espera_real)
        espera = min(espera * 2, 60)
    resp.raise_for_status()
    return resp


TERMINO_GENERICO = {'RESTAURANT': 'restaurante'}

# Nombres tal como los guarda TripAdvisor internamente (confirmado a mano en
# tripadvisor.com), distintos del nombre oficial que usa limites_municipales.
NOMBRES_CORTOS = {
    'San Cristóbal de La Laguna': 'La Laguna',
    'Güímar': 'Guimar',
    'Guía de Isora': 'Guia de Isora',
    'Santa Úrsula': 'Santa Ursula',
    'Vilaflor de Chasna': 'Vilaflor',
    # El Rosario: sin alias posible -- no tiene pagina de destino propia en
    # TripAdvisor, probablemente absorbido dentro del area metropolitana de
    # La Laguna. Limitacion documentada, no se fuerza ningun nombre.
}


def nombre_busqueda(municipio):
    return NOMBRES_CORTOS.get(municipio, municipio)


def buscar_ubicaciones(municipio, categoria, max_paginas=10):
    resultados_totales = []
    pagina = 1
    usar_fallback = (categoria == 'HOTEL')
    while pagina <= max_paginas:
        if usar_fallback:
            params = {
                'query': nombre_busqueda(municipio),
                'category': categoria,
                'country_code': 'ES',
                'size': 20,
                'page': pagina,
            }
        else:
            params = {
                'query': TERMINO_GENERICO.get(categoria, categoria.lower()),
                'geo_name': nombre_busqueda(municipio),
                'category': categoria,
                'size': 20,
                'page': pagina,
            }
        try:
            resp = peticion_con_reintentos(BASE_URL + '/locations/search', params)
        except Exception as error:
            if not usar_fallback:
                print('    geo_name fallo (', error, '), cambiando al metodo de texto para el resto de paginas...')
                usar_fallback = True
                params = {'query': nombre_busqueda(municipio), 'category': categoria, 'country_code': 'ES', 'size': 20, 'page': pagina}
                resp = peticion_con_reintentos(BASE_URL + '/locations/search', params)
            else:
                print('    fallo en pagina', pagina, '- me quedo con lo ya conseguido en paginas anteriores')
                break
        resultados_pagina = resp.json().get('data', [])
        if not resultados_pagina:
            break
        resultados_totales.extend(resultados_pagina)
        if len(resultados_pagina) < 20:
            break
        pagina += 1
        time.sleep(0.5)
    return resultados_totales


def obtener_resenas(location_id, idioma='es'):
    params = {'language': idioma, 'size': 20}
    resp = peticion_con_reintentos(
        BASE_URL + '/locations/' + str(location_id) + '/reviews', params
    )
    return resp.json().get('data', [])


def esta_en_tenerife(resultado):
    coords = (resultado.get('location', {}) or {}).get('coordinates') or {}
    lat, lon = coords.get('latitude'), coords.get('longitude')
    if lat is None or lon is None:
        return False
    return Point(float(lon), float(lat)).within(tenerife_union)

## Paso 5 — Recorrer los 31 municipios, validando geografía ANTES de pedir reseñas

Prueba primero con `MUNICIPIOS[:3]` si quieres confirmar que todo va bien antes de lanzar los 31 completos. Recuerda comprobar el cupo diario en "Usage and access" antes de lanzar la tanda completa.

In [ ]:
import pandas as pd

MUNICIPIOS = pd.read_sql('SELECT etiqueta FROM silver.limites_municipales;', engine)['etiqueta'].tolist()
# MUNICIPIOS = MUNICIPIOS[:3]  # descomenta para probar con pocos municipios primero

CATEGORIAS = ['HOTEL', 'RESTAURANT']
LIMITE_DIARIO_SEGURO = 900

ubicaciones_raw = []
resenas_raw = []
vistos = set()
descartados_fuera_tenerife = 0
parar = False

for municipio in MUNICIPIOS:
    if parar:
        break
    for categoria in CATEGORIAS:
        if peticiones_realizadas >= LIMITE_DIARIO_SEGURO:
            print()
            print('LIMITE DE SEGURIDAD ALCANZADO (' + str(peticiones_realizadas) + ' peticiones). Parando aqui.')
            print('Continuad manana desde:', municipio, '-', categoria)
            parar = True
            break

        print('Buscando', categoria, 'en', municipio, '...')
        try:
            resultados = buscar_ubicaciones(municipio, categoria)
        except Exception as error:
            print('  aviso: fallo la busqueda -', error)
            continue

        validos = [r for r in resultados if esta_en_tenerife(r)]
        descartados_fuera_tenerife += len(resultados) - len(validos)
        print('  ->', len(resultados), 'encontrados,', len(validos), 'dentro de Tenerife')

        for resultado in validos:
            if peticiones_realizadas >= LIMITE_DIARIO_SEGURO:
                print('  limite alcanzado a mitad de', municipio, categoria, '- paro aqui')
                parar = True
                break

            loc = resultado.get('location', {})
            location_id = loc.get('id')
            if location_id is None or location_id in vistos:
                continue
            vistos.add(location_id)

            registro = dict(resultado)
            registro['municipio_busqueda'] = municipio
            registro['categoria_busqueda'] = categoria
            ubicaciones_raw.append(registro)

            time.sleep(0.5)

            try:
                resenas = obtener_resenas(location_id)
            except Exception as error:
                print('  aviso: fallaron las resenas de', location_id, '-', error)
                resenas = []

            for resena in resenas:
                resenas_raw.append({'location_id': location_id, 'resena_raw': resena})

            time.sleep(0.5)

        if parar:
            break

print()
print('Ubicaciones validas (dentro de Tenerife):', len(ubicaciones_raw))
print('Descartadas por estar fuera de Tenerife:', descartados_fuera_tenerife)
print('Resenas:', len(resenas_raw))
print('Peticiones usadas:', peticiones_realizadas)

## Paso 6 — Subir el crudo a Bronze

In [ ]:
import json
from datetime import date

fecha = date.today().isoformat()
nombre_blob_ubicaciones = 'tripadvisor/ubicaciones_raw_' + fecha + '.json'
nombre_blob_resenas = 'tripadvisor/resenas_raw_' + fecha + '.json'

container_client.upload_blob(name=nombre_blob_ubicaciones, data=json.dumps(ubicaciones_raw), overwrite=True)
container_client.upload_blob(name=nombre_blob_resenas, data=json.dumps(resenas_raw), overwrite=True)

print('Subido a bronze:', nombre_blob_ubicaciones)
print('Subido a bronze:', nombre_blob_resenas)

## Paso 7 — Limpiar, validar de nuevo, y reconstruir Silver por completo

Borra todo lo anterior en `silver.tripadvisor_ubicaciones`/`tripadvisor_resenas` y lo sustituye por esta ejecución — pensado para ser la fuente única de verdad, no para ir acumulando ejecuciones parciales.

In [ ]:
def valor_traducido(items, idioma='es'):
    if not items:
        return None
    for item in items:
        if item.get('language') == idioma:
            return item.get('value')
    for item in items:
        if item.get('primary'):
            return item.get('value')
    return items[0].get('value')


ubicaciones_raw_leidas = json.loads(
    container_client.get_blob_client(nombre_blob_ubicaciones).download_blob().readall()
)
resenas_raw_leidas = json.loads(
    container_client.get_blob_client(nombre_blob_resenas).download_blob().readall()
)

filas_ubicaciones = []
for registro in ubicaciones_raw_leidas:
    loc = registro.get('location', {})
    coords = loc.get('coordinates') or {}
    direcciones = loc.get('addresses') or [{}]
    rating_info = (loc.get('traveler_ratings') or {}).get('overall') or {}

    filas_ubicaciones.append({
        'location_id': loc.get('id'),
        'nombre': valor_traducido(loc.get('names')),
        'categoria': registro.get('categoria_busqueda'),
        'municipio_busqueda': registro.get('municipio_busqueda'),
        'direccion': direcciones[0].get('formatted'),
        'rating': rating_info.get('rating'),
        'num_resenas': rating_info.get('count'),
        'nivel_precio': loc.get('price_level'),
        'latitud': coords.get('latitude'),
        'longitud': coords.get('longitude'),
    })

df_ubicaciones = pd.DataFrame(filas_ubicaciones).dropna(subset=['latitud', 'longitud'])
geometry = [
    Point(float(lon), float(lat))
    for lon, lat in zip(df_ubicaciones['longitud'], df_ubicaciones['latitud'])
]
gdf_ubicaciones = gpd.GeoDataFrame(df_ubicaciones, geometry=geometry, crs='EPSG:4326')

antes = len(gdf_ubicaciones)
gdf_ubicaciones = gdf_ubicaciones[gdf_ubicaciones.within(tenerife_union)]
print('Segunda pasada de validacion geografica: descartadas', antes - len(gdf_ubicaciones), '(deberia ser 0, ya se filtro antes)')

gdf_ubicaciones = gdf_ubicaciones.to_crs(epsg=32628)

with engine.begin() as conn:
    conn.execute(text('DELETE FROM silver.tripadvisor_resenas'))
    conn.execute(text('DELETE FROM silver.tripadvisor_ubicaciones'))
print('Tablas anteriores borradas por completo.')

gdf_ubicaciones.to_postgis(
    'tripadvisor_ubicaciones', engine, schema='silver', if_exists='append', index=False
)
with engine.begin() as conn:
    conn.execute(text(
        'CREATE INDEX IF NOT EXISTS idx_tripadvisor_ubicaciones_geometry '
        'ON silver.tripadvisor_ubicaciones USING GIST (geometry)'
    ))
print('-> silver.tripadvisor_ubicaciones reconstruida:', len(gdf_ubicaciones), 'filas')

filas_resenas = []
for item in resenas_raw_leidas:
    resena = item.get('resena_raw', {})
    filas_resenas.append({
        'review_id': resena.get('id'),
        'location_id': item.get('location_id'),
        'rating': resena.get('rating'),
        'titulo': valor_traducido(resena.get('title')),
        'texto': valor_traducido(resena.get('text')),
        'fecha_publicacion': resena.get('publish_ts'),
        'fecha_viaje': resena.get('travel_date'),
        'tipo_viaje': resena.get('trip_type'),
        'usuario': (resena.get('user') or {}).get('username'),
    })

df_resenas = pd.DataFrame(filas_resenas)
df_resenas.to_sql('tripadvisor_resenas', engine, schema='silver', if_exists='append', index=False)
print('-> silver.tripadvisor_resenas reconstruida:', len(df_resenas), 'filas')

## Paso 8 — Verificar

In [ ]:
with engine.connect() as conn:
    resumen = pd.read_sql('''
        SELECT municipio_busqueda,
               COUNT(*) FILTER (WHERE categoria = 'HOTEL') AS hoteles,
               COUNT(*) FILTER (WHERE categoria = 'RESTAURANT') AS restaurantes
        FROM silver.tripadvisor_ubicaciones
        GROUP BY municipio_busqueda
        ORDER BY municipio_busqueda
    ''', conn)
    total = pd.read_sql('SELECT COUNT(*) AS total FROM silver.tripadvisor_ubicaciones;', conn)
    num_municipios = pd.read_sql('SELECT COUNT(DISTINCT municipio_busqueda) AS n FROM silver.tripadvisor_ubicaciones;', conn)
    total_resenas = pd.read_sql('SELECT COUNT(*) FROM silver.tripadvisor_resenas;', conn)
    verificacion_geo = pd.read_sql('''
        SELECT COUNT(*) AS fuera_de_tenerife FROM silver.tripadvisor_ubicaciones u
        WHERE NOT ST_Within(u.geometry, (SELECT ST_Union(geometry) FROM silver.limites_municipales))
    ''', conn)

print('Total de filas:', total['total'][0], '(esperado: 750)')
print('Municipios distintos:', num_municipios['n'][0], '(esperado: 30 de 31 -- El Rosario no aparece, ver notas)')
print('Total resenas:', total_resenas.iloc[0, 0])
print('Filas fuera de Tenerife (debe ser 0):', verificacion_geo['fuera_de_tenerife'][0])
display(resumen)

## Notas — decisiones y limitaciones conocidas

- **HOTEL + `geo_name` (y también `address`) da un 503 reproducible** en el servidor de Terra, confirmado con múltiples municipios y parámetros distintos. No tiene solución desde nuestro lado — para HOTEL se usa búsqueda de texto (nombre del municipio + `country_code='ES'`), que puede infravalorar la oferta real en municipios donde los hoteles no incluyen el nombre del municipio en su denominación comercial. Confirmado en Güímar: 3 hoteles reales verificados a mano en tripadvisor.com frente a 1 encontrado por la API.
- **El Rosario no tiene página de destino propia en TripAdvisor** (confirmado a mano) — cualquier búsqueda de su nombre encuentra solo homónimos de fuera de Tenerife, que la validación geográfica descarta correctamente. Se queda sin datos, y es una limitación real, no un fallo del código.
- **La validación geográfica (`esta_en_tenerife`) es imprescindible**, no opcional: sin ella, la búsqueda de texto sin `geo_name` para HOTEL cuela resultados de cualquier parte del mundo que compartan nombre con un municipio de Tenerife (confirmado: hoteles de Italia, Guatemala, Argentina, y de Gran Canaria colándose con nombres como "Arona", "Candelaria", "El Rosario", "Santa Cruz").
- Se probó también un endpoint de búsqueda por coordenadas (`catalog/locations/nearby`) como alternativa para HOTEL — no respetó el filtro de categoría solicitado (devolvió atracciones), así que se descartó.